### AES (Advanced Encryption Standard) Encryption in Python

##### 1. What is Encryption

Encryption is a process where we convert readable data into an unreadable format so that only authorized users can access or understand the data.  
The main goal of encryption is to protect sensitive information like passwords, API keys, financial data, and personal records.  

When data is encrypted, it looks like random characters, but it can be converted back using a secret key.

##### Example:
- Original data: `MyPassword123`
- Encrypted data: `b'\x8f\xa3\x91...'` (random bytes)

**Important idea:**
- **Encryption → Hide data**
- **Decryption → Get original data back**

##### Binary Byte Format

Data stored as raw bytes using hexadecimal values, where each byte represents 8 bits (0s and 1s). Instead of readable text, data is written like `\x..` which represents actual binary values in hex form.

1. Example 1

      ```python
      b"\x41"
      ```

      | Step | Value |
      |------|-------|
      | Hex | `41` |
      | Decimal | `65` |
      | ASCII | `'A'` |

      **Output:** `b'A'`

2. Example 2

      ```python
      b"\x48\x65\x6c\x6c\x6f"
      ```

      | Hex | Character |
      |-----|-----------|
      | `\x48` | H |
      | `\x65` | e |
      | `\x6c` | l |
      | `\x6c` | l |
      | `\x6f` | o |

      **Output:** `b'Hello'`

##### How It Works Internally

- Computers store everything in binary (0s and 1s)
- Hex notation (`\x..`) is used because it is more readable than raw binary

**Example:**

| Representation | Value |
|----------------|-------|
| Character | `H` |
| Binary | `01001000` |
| Hex | `\x48` |


##### 2. What is AES

AES stands for **Advanced Encryption Standard**, and it is one of the most widely used encryption techniques in the world.  

It is a **symmetric encryption algorithm**, which means the same key is used for both encryption and decryption processes.  

AES is considered **fast, secure, and efficient**, which is why it is used in:
- Banking systems  
- Cloud storage  
- Secure APIs  

**Interview-friendly line:**  
AES is a symmetric encryption algorithm used to securely encrypt and decrypt data using a shared secret key.


##### 3. Why AES is Important for Data Engineers

In data engineering, we often handle sensitive datasets such as:
- User data  
- Transaction logs  
- API credentials  

These must be protected from unauthorized access.
AES helps in ensuring that even if data is leaked or intercepted, it cannot be read without the key.

##### Common use cases:
- Secure data pipelines  
- Cloud storage encryption (AWS S3, GCP buckets)  
- Database encryption  
- Data transfer over APIs  

##### 4. Core Concepts of AES

##### 4.1 Key (Secret Key)

A key is a secret piece of information used to encrypt and decrypt data.  
The strength of encryption depends heavily on how secure the key is.

AES supports three key sizes:

| Key Size            | Security Level        | Usage                |
|--|-|-|
| 128-bit (16 bytes) | Good security        | Common usage         |
| 192-bit (24 bytes) | Better security      | Less common          |
| 256-bit (32 bytes) | Very strong security | High-security systems|

**Important:**
- Larger key size → More secure but slightly slower  
- Never share the key publicly  


##### 4.2 Block Size

AES works on a fixed block size of **16 bytes (128 bits)**.  

This means data is processed in chunks of 16 bytes at a time.  
If data is not exactly 16 bytes, we need to add padding.


##### 4.3 Padding
Padding is used when the input data size is not a multiple of 16 bytes.  
It adds extra bytes so that the data fits into AES block size.

##### Example:
- Data: `"Hello"` → 5 bytes  
- After padding → becomes 16 bytes  

**Without padding → encryption will fail**


##### 4.4 Modes of AES

AES does not directly encrypt large data, so it uses different modes of operation.

- **ECB (Electronic Codebook Mode)**
  - Encrypts each block independently  
  - Same input → same output  
  - Not secure because patterns are visible  

**Never use ECB in real applications**


- **CBC (Cipher Block Chaining Mode)**
  - Each block depends on the previous block’s encryption  
  - Uses IV (Initialization Vector)  
  - More secure than ECB, but still not the best  


- **GCM (Galois/Counter Mode)**
  - Provides:
    - Encryption  
    - Authentication (checks data integrity)  
  - Fast and secure  
  - Used in modern applications  

**What is a Cipher**: A cipher is a method or algorithm used to convert normal data into encrypted data and also back to original data.
It is the actual technique behind encryption and decryption.


**Interview tip:**  
GCM is preferred because it ensures both confidentiality and data integrity.


##### 4.5 IV (Initialization Vector)

IV is a random value used in encryption to make output different every time.  

Even if same data and key are used, IV ensures different encrypted output.  
It is mainly used in CBC and GCM modes.

**Important:**
- IV is not secret, but it must be random  
- Never reuse IV with same key  

##### 5. Python Library for AES — PyCryptodome

- **PyCryptodome** is a popular Python library used for cryptography operations
- It provides simple and reliable functions to implement AES encryption

**Installation:**

```bash
pip install pycryptodome
```


##### 6. Basic AES Encryption Flow (Concept)

Steps to encrypt data using AES:

1. Take the **input data**
2. Generate a **secret key**
3. Choose an **AES mode** (CBC or GCM)
4. Add **padding** if data size is not a multiple of 16 bytes
5. **Encrypt** the data using the cipher
6. **Store IV/nonce** alongside ciphertext for later decryption


##### 7. Scenario 1 — Real Data Engineering Use Case

**Scenario:**
You are building a data pipeline that stores user API keys in a database.
You do not want to store them in plain text.

**Without Encryption:**

```python
API_KEY = "abcd1234"
```

→ Anyone with database access can read it directly

**With AES Encryption:**

- Store the **encrypted value** in the database instead of plain text
- **Decrypt only when** the value is actually needed at runtime

##### 8. Implementation — AES CBC Mode

**Encryption Code:**

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad

data = b"MySecretAPIKey"

key = get_random_bytes(16)

cipher = AES.new(key, AES.MODE_CBC)

ciphertext = cipher.encrypt(pad(data, AES.block_size))

iv = cipher.iv

print("Encrypted:", ciphertext)
print("IV:", iv)

Encrypted: b'\xb8w\x0cIG\x11@/\x06O\x0bB\xa1[\xc3.'
IV: b'\xecs\xd7`\xbe\x1c\\R\xaf\xc9\xc7(\x8a\x9bWR'


**Decryption Code:**

In [10]:
from Crypto.Util.Padding import unpad

cipher_dec = AES.new(key, AES.MODE_CBC, iv=iv)

original = unpad(cipher_dec.decrypt(ciphertext), AES.block_size)

print("Decrypted:", original)

Decrypted: b'MySecretAPIKey'


##### 9. Code Explanation — Key Functions

- `get_random_bytes(16)` → generates a **secure random 16-byte key**
- `AES.new(key, AES.MODE_CBC)` → creates a new **cipher object** in CBC mode
- `pad()` → ensures data length is a **multiple of 16 bytes** before encryption
- `encrypt()` → converts plaintext data into **encrypted ciphertext**
- `iv` → must be stored and passed during **decryption**
- `unpad()` → **removes the extra padding** added before encryption


##### 10. Example 2 — Encrypting User Email Data

**Scenario:** You want to store a user's email address securely in a database.

**Input:**

```python
data = b"user@email.com"
```

**After Encryption:**

```text
b'\x8f\xa1\xbb...'
```

→ This encrypted value is stored in the database instead of the plain email


##### 11. Important Observations

- **Same data + same key → different output** every time, because IV changes each run
- Encrypted data is always in **bytes format**, not string
- Always store both:
  - **Key** — store securely (e.g., environment variable or key vault)
  - **IV** — store alongside the ciphertext for decryption

##### 12. AES GCM Mode (Best Practice for Real Use)

- Advanced Encryption Standard - Galois/Counter Mode (AES GCM)
- **AES-GCM (Galois/Counter Mode)** is a modern and secure encryption mode used in real-world systems
- It provides both:
  - **Confidentiality** — encrypts the data so it cannot be read
  - **Integrity** — verifies that data has not been tampered with
- If someone modifies the encrypted data, **decryption will fail automatically**

**Simple Understanding:**

- GCM not only **hides the data** but also **verifies that data is not changed**

##### 13. Why GCM is Better than CBC

- In **CBC mode**, encryption is secure but it **does not verify data integrity** — modified data may still decrypt
- **GCM mode** adds an extra security layer called a **tag**, which ensures data authenticity
- GCM is **faster** and widely used in APIs, HTTPS, and cloud systems
- A **nonce** is a unique random value used in GCM mode that ensures even with the same data and key, the encrypted output is different every time.

##### 14. AES GCM Implementation in Python

**Encryption Code:**

In [6]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

data = b"UserSensitiveData"

key = get_random_bytes(16)

cipher = AES.new(key, AES.MODE_GCM)

ciphertext, tag = cipher.encrypt_and_digest(data)

nonce = cipher.nonce

print("Encrypted:", ciphertext)
print("Tag:", tag)
print("Nonce:", nonce)

Encrypted: b'\xdaGv\x16\x90\x0b\xe9\xba\x08]_\xdf\x1a\x1a\x8c\xdf0'
Tag: b'V\xf5-\xc4\x07@\xebFM\xf1\xa1\x01$\xe9!\x8b'
Nonce: b'\x18Bm\xb4Wg\x0c\t\xfb@<\xb0\x05\x13\xed4'


**Decryption Code:**

In [7]:
cipher_dec = AES.new(key, AES.MODE_GCM, nonce=nonce)

original = cipher_dec.decrypt_and_verify(ciphertext, tag)

print("Decrypted:", original)

Decrypted: b'UserSensitiveData'


**Output Example:**

```text
Encrypted: b'\x8f\xa2...'
Tag: b'\x91\xbc...'
Nonce: b'\x11\x2a...'
Decrypted: b'UserSensitiveData'
```

##### 15. Understanding GCM Components

- **ciphertext** → the encrypted form of original data
- **tag** → used to verify the integrity of data during decryption
- **nonce** → similar to IV, ensures randomness in each encryption

**Important:**

- All three — **ciphertext, tag, and nonce** — are required for successful decryption

##### 16. Scenario 2 — Secure Data Pipeline

**Scenario:**
You are building a pipeline that sends user data from one service to another API.

**Problem:**
- Data can be **intercepted during transfer**

**Solution using AES-GCM:**
- **Encrypt data** before sending from sender side
- **Decrypt at receiver** side upon arrival
- If data is **modified in transit** → decryption fails automatically

**Implementation Idea:**

```python
# Sender side
encrypted_data = encrypt(data)

# Receiver side
original_data = decrypt(encrypted_data)
```

**This ensures:**
- Data is **hidden** from interceptors
- Data is **not modified** without detection


##### 17. Base64 Encoding

- AES output is in **binary (bytes) format**
- Databases and APIs usually require **string format**
- **Base64 encoding** is used to convert bytes → string safely

**Example:**

```python
import base64

encoded = base64.b64encode(ciphertext)
print(encoded)

decoded = base64.b64decode(encoded)
```

**Why Base64 is Needed:**
- Easy to **store in a database** without corruption
- Easy to **send via APIs** in JSON format


##### 18. Scenario 3 — Storing Encrypted Data in Database

**Scenario:**
You want to store encrypted user data in a SQL table.

**Without Base64:**

```text
b'\x8f\xa1\xbb...'
```

→ Not readable and not safe for direct database storage

**With Base64:**

```text
j6G7k29...
```

→ Clean string format, safe to store and retrieve


##### 19. Key Management

- **Key** is the most sensitive part of the entire encryption system
- If the key is **leaked → encryption becomes completely useless**

**Bad Practices:**
- Hardcoding key directly inside source code
- Storing key in a GitHub repository
- Sharing key via email or chat

**Good Practices:**
- Store key in **environment variables**
- Use dedicated **secret managers** (AWS Secrets Manager, GCP Secret Manager)
- **Rotate keys** regularly to minimize exposure risk
- **Restrict access** to keys using role-based permissions

**Example — Reading Key from Environment Variable:**

```python
import os

key = os.getenv("AES_KEY")
```

##### 20. Full Real-World Flow (End-to-End)

**Data Engineering Flow:**

1. Data is **collected from source**
2. Sensitive fields are **encrypted using AES**
3. Encrypted data is **stored in database** in Base64 format
4. When needed → **decrypt the data**
5. Use **decrypted data** for processing or analytics


##### 21. Common Real-World Mistakes

- **Reusing the same nonce/IV** across multiple encryptions
- **Not verifying the tag** in GCM mode after decryption
- Storing encrypted data **without Base64 encoding**
- Using **weak or predictable keys** instead of secure random keys
- **Not handling exceptions** that occur during decryption failures


##### 22. Best Practices Summary

- Always prefer **AES-GCM mode** over CBC for new implementations
- Use **secure random keys** generated via `get_random_bytes()`
- **Store keys outside code** — use environment variables or secret managers
- Use **Base64 encoding** for storing or transmitting encrypted data
- **Never reuse IV/nonce** with the same key
- Always **verify integrity using the tag** in GCM mode

##### 25. Example — Complete Working Mini Project

**Scenario:** Encrypt and store a user password securely using AES-GCM with Base64

**Encryption Code:**

In [11]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64

data = b"MyPassword123"

key = get_random_bytes(16)

cipher = AES.new(key, AES.MODE_GCM)

ciphertext, tag = cipher.encrypt_and_digest(data)

nonce = cipher.nonce

encoded_cipher = base64.b64encode(ciphertext)
encoded_nonce = base64.b64encode(nonce)
encoded_tag = base64.b64encode(tag)

print(encoded_cipher)
print(encoded_nonce)
print(encoded_tag)

b'9xh8SXMaggJ1p7IHLA=='
b'i1QFcwUCWEKDf+5Tye0Ldg=='
b'njmUk6pZyEqHdtltBI2JMA=='


**Decryption Code:**

In [12]:
cipher_dec = AES.new(key, AES.MODE_GCM, nonce=base64.b64decode(encoded_nonce))

original = cipher_dec.decrypt_and_verify(
    base64.b64decode(encoded_cipher),
    base64.b64decode(encoded_tag)
)

print(original)

b'MyPassword123'


#### Exercise

1. What is encryption and why is it used?

    Encryption is the process of converting readable data into an unreadable format using an algorithm and a key, so that only authorized users can access it. It is used to protect sensitive information like passwords, personal data, and API keys from unauthorized access.

2. What is AES and what type of encryption is it?

    AES (Advanced Encryption Standard) is a widely used encryption algorithm that is used to secure data. It is a symmetric type of encryption, meaning the same key is used for both encryption and decryption.

3. What is the block size of AES?

    AES has a fixed block size of 16 bytes (128 bits), meaning it processes data in chunks of 16 bytes at a time.

    It is fixed because the internal design of AES is built specifically for 128-bit blocks, which makes the algorithm efficient, secure, and standardized across all systems.

4. What is padding and why is it required?

    Padding is the process of adding extra bytes to data so its length becomes a multiple of 16 bytes, which is required by AES block size.

    It is needed because AES can only encrypt data in fixed 16-byte blocks, so if the data is smaller or not aligned, padding ensures it fits correctly for encryption.

1. What is IV and why is it important?

    IV (Initialization Vector) is a random value used along with the key during encryption to add randomness to the process.

    It is important because it ensures that even if the same data and same key are used, the encrypted output will be different each time, which improves security and prevents pattern detection.

2. Why is ECB mode not considered secure?

    ECB (Electronic Codebook) mode is not considered secure because it encrypts each block independently, so same input data always produces the same encrypted output.

    This creates visible patterns in the ciphertext, which attackers can analyze and use to guess the original data, making it insecure for real-world use.

3. Write Python code to encrypt a message using AES CBC mode

In [13]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad

data = b"MySecretKey"
key = get_random_bytes(16)

cipher = AES.new(key, AES.MODE_CBC)
ciphertext = cipher.encrypt(pad(data, AES.block_size))

print(ciphertext)

b'\xb9v_\x1d\x0b\x00\xbcU\x1boP\x1c\x1a\x1a\xf7\xcf'


4. What happens if padding is not applied before encryption?

    If padding is not applied, and the data length is not a multiple of 16 bytes, AES encryption will fail and raise an error because it cannot process incomplete blocks.

1. You are storing API keys in a database — how will you secure them using AES?

    To secure API keys in a database using AES, I would encrypt the API keys before storing them and decrypt them only when needed. I would use AES-GCM mode for better security, generate a random key and nonce, and store the encrypted data along with the nonce and tag in the database. The encryption key would be kept securely in environment variables or a secret manager, not inside the code. This ensures that even if the database is compromised, the API keys cannot be read without the key.

2. Explain the full AES encryption flow as used in a data pipeline

    In a data pipeline, AES encryption is used to protect sensitive data during storage and transfer.

    First, data is collected from the source, and any sensitive fields like API keys or user information are identified. These fields are then converted into byte format and encrypted using AES (usually GCM mode) with a secure key and a random nonce. The output of encryption includes ciphertext, nonce, and tag.

    Next, the encrypted data is encoded (usually using Base64) and stored in a database or sent through APIs. The encryption key is stored securely in environment variables or a secret manager, not in the code.

    When the data needs to be used, it is fetched from storage, decoded from Base64, and then decrypted using the same key, nonce, and tag. If the data was modified, decryption will fail in GCM mode, ensuring data integrity.

    This process ensures that sensitive data remains secure throughout the entire pipeline, even if storage or transmission is compromised.

#### AES-GCM Encryption in Python (Real world implementation)

In [25]:
import base64  # used to convert binary data to string format (for storage/API)

from Crypto.Cipher import AES  # AES encryption module
from Crypto.Protocol.KDF import PBKDF2  # used to derive strong key from password
from Crypto.Random import get_random_bytes  # used internally for random nonce


# user-provided password (this is not directly used as AES key)
password = "key1"

# salt adds randomness and improves security of key generation
salt = b"salt123"


# function to generate secure 32-byte key using PBKDF2
def get_key():
    # PBKDF2 converts password + salt → strong encryption key
    return PBKDF2(password, salt, dkLen=32, count=100000)


# encryption function
def encrypt(data):
    key = get_key()  # generate AES key

    # create AES cipher in GCM mode (secure mode with authentication)
    cipher = AES.new(key, AES.MODE_GCM)

    # encrypt data and generate authentication tag
    ciphertext, tag = cipher.encrypt_and_digest(data.encode())

    # return encrypted data in Base64 format (safe for storage/transfer)
    return {
        "ciphertext": base64.b64encode(ciphertext).decode(),  # encrypted data
        "nonce": base64.b64encode(cipher.nonce).decode(),     # random nonce
        "tag": base64.b64encode(tag).decode()                 # integrity check
    }


# decryption function
def decrypt(enc_data):
    key = get_key()  # generate same key again

    # create AES cipher with same nonce used during encryption
    cipher = AES.new(
        key,
        AES.MODE_GCM,
        nonce=base64.b64decode(enc_data["nonce"])
    )

    # decrypt data and verify integrity using tag
    decrypted = cipher.decrypt_and_verify(
        base64.b64decode(enc_data["ciphertext"]),
        base64.b64decode(enc_data["tag"])
    )

    # convert bytes back to string
    return decrypted.decode()


# test the encryption and decryption
encrypted = encrypt("anu_andy")
print("Encrypted:", encrypted)

decrypted = decrypt(encrypted)
print("Decrypted:", decrypted)

Encrypted: {'ciphertext': 'GqoVHOFPkWU=', 'nonce': 'HrUXSRVffUA6zqJnRn+5mQ==', 'tag': 'ZNJasBMenhQoxV3FX3VW7Q=='}
Decrypted: anu_andy


##### Code Explanation

##### 1. Overview

This code demonstrates secure encryption and decryption using AES-GCM mode.  
It includes:
- Key generation using PBKDF2  
- Encryption with AES-GCM  
- Base64 encoding for storage  
- Decryption with integrity verification  

##### 2. Key Generation (PBKDF2)

- Instead of using a simple password directly, we use PBKDF2 to generate a strong key  
- It takes:
  - Password
  - Salt
  - Iterations (100000)
- Output is a 32-byte key (AES-256)

##### 3. Encryption Flow

Step-by-step:
1. Generate key using PBKDF2  
2. Create AES cipher in GCM mode  
3. Convert data to bytes using `.encode()`  
4. Encrypt data using `encrypt_and_digest()`  
5. Get:
   - Ciphertext (encrypted data)
   - Tag (for integrity check)
   - Nonce (random value for security)
6. Convert all outputs to Base64 for safe storage  

##### 4. Decryption Flow

Step-by-step:
1. Generate same key using PBKDF2  
2. Decode Base64 values (ciphertext, nonce, tag)  
3. Create AES cipher using same nonce  
4. Decrypt using `decrypt_and_verify()`  
5. Convert result back to string  

##### 5. Important Concepts

##### AES-GCM Mode
- Provides both encryption and authentication  
- Ensures data is not modified  

##### Nonce
- Random value generated during encryption  
- Must be same during decryption  

##### Tag
- Used to verify data integrity  
- If data is changed → decryption fails  

##### Base64 Encoding
- Converts binary data into string format  
- Required for storing in DB or sending via API  

##### 6. Security Highlights

- Uses strong key derivation (PBKDF2)  
- Uses AES-GCM (secure mode)  
- Uses random nonce  
- Verifies integrity using tag  

##### 7. Final Understanding

This code shows a real-world secure encryption flow where:
- Data is encrypted before storage  
- Stored safely in encoded format  
- Decrypted only when needed  
- Any tampering is detected automatically  